# Building a Dynamic Amazon Lex Chatbot with AWS Lambda and Third-Party API Integration

## Lab Overview

**Scenario:** You are a Machine Learning Engineer at GlobalTech Solutions. Your team is developing an intelligent chatbot using Amazon Lex for a travel assistance application. The chatbot should answer questions about various countries, like *"What is the capital of France?"* or *"What is the currency of India?"* You will integrate the chatbot with a third-party API to retrieve real-time data about countries.

**What you will build:**
- An AWS Lambda function with Node.js logic to call third-party APIs
- An Amazon Lex V2 bot with 6 intents and custom slot types
- End-to-end integration tested programmatically via boto3

**Lab Steps:**
1. Setup & Prerequisites
2. Create the Lambda Function (with code deployment)
3. Configure Environment Variables & Lambda Settings
4. Create Custom Slot Types in Lex
5. Create the Lex Bot & All 6 Intents
6. Build the Bot
7. Test the Bot Programmatically
8. Cleanup Resources

---
> **Note:** This notebook replicates the full console-based lab using Python (boto3). Run cells **top to bottom** in order.

## Cell 1 — Install Dependencies

We need `boto3` (AWS SDK) and `zipfile` (built-in) to package our Lambda function. The `requests` library is used for direct API testing at the end.

In [ ]:
# Install/upgrade boto3 to the latest version to ensure Lex V2 support
import subprocess
subprocess.run(["pip", "install", "--upgrade", "boto3", "--quiet"], check=True)
print("boto3 installed/upgraded successfully.")

## Cell 2 — Imports & Configuration

Set your AWS region and your API keys here. The OpenWeatherMap key is **required**; the News API key is optional.

In [ ]:
import boto3
import json
import zipfile
import io
import time
import os

# ─────────────────────────────────────────────────────────────
# CONFIGURATION — Edit these values before running the notebook
# ─────────────────────────────────────────────────────────────

AWS_REGION = "us-east-1"               # Change to your preferred region
OPENWEATHER_API_KEY = "YOUR_OPENWEATHER_API_KEY_HERE"   # Required
NEWS_API_KEY        = "YOUR_NEWS_API_KEY_HERE"           # Optional

BOT_NAME            = "TravelIntelligenceBot"
LAMBDA_FUNCTION_NAME = "TravelIntelligenceBot"
LOCALE_ID           = "en_US"

# ─────────────────────────────────────────────────────────────
# boto3 Clients
# ─────────────────────────────────────────────────────────────
session     = boto3.Session(region_name=AWS_REGION)
iam_client  = session.client("iam")
lambda_client = session.client("lambda")
lex_client  = session.client("lexv2-models")
lex_runtime = session.client("lexv2-runtime")
sts_client  = session.client("sts")

# Fetch Account ID (used when constructing ARNs)
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
print(f"✅ Connected to AWS Account: {ACCOUNT_ID}  |  Region: {AWS_REGION}")

## Cell 3 — Create IAM Role for Lambda

The Lambda function needs an execution role with permissions to:
- Write logs to CloudWatch
- Be invoked by Amazon Lex

> **What happens here:** We create a role called `TravelIntelligenceBotLambdaRole` with a trust policy that allows Lambda to assume it, then attach the `AWSLambdaBasicExecutionRole` managed policy.

In [ ]:
LAMBDA_ROLE_NAME = "TravelIntelligenceBotLambdaRole"

# Trust policy — allows Lambda service to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role_resp = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for TravelIntelligenceBot Lambda function"
    )
    LAMBDA_ROLE_ARN = role_resp["Role"]["Arn"]
    print(f"✅ Created Lambda role: {LAMBDA_ROLE_ARN}")
except iam_client.exceptions.EntityAlreadyExistsException:
    # Role already exists — just fetch its ARN
    LAMBDA_ROLE_ARN = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    print(f"ℹ️  Role already exists. Using ARN: {LAMBDA_ROLE_ARN}")

# Attach the AWS-managed basic execution policy (CloudWatch Logs access)
iam_client.attach_role_policy(
    RoleName=LAMBDA_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
)

# Wait for IAM role propagation (IAM changes take a few seconds to be globally consistent)
print("⏳ Waiting 15 seconds for IAM role propagation...")
time.sleep(15)
print("✅ IAM role ready.")

## Cell 4 — Write the Lambda Function Code

This is the full Node.js 22.x handler that the original lab deploys. It handles all 6 intents:

| Intent | API Used |
|---|---|
| CountryInfo | restcountries.com |
| WeatherInfo | openweathermap.org |
| CurrencyConversion | exchangerate-api.com (free) |
| TravelRecommendations | Built-in logic |
| FlightStatus | Simulated (no free real-time API) |
| TravelItinerary | Built-in logic |

The code is written to a local file `index.mjs`, then zipped and uploaded to Lambda.

In [ ]:
LAMBDA_CODE = r"""
import https from 'https';

// ─── Utility: make HTTPS GET request and return parsed JSON ───
function httpsGet(url) {
  return new Promise((resolve, reject) => {
    const options = {
      headers: {
        'User-Agent': 'Mozilla/5.0 (compatible; LexLab/1.0)',
        'Accept': 'application/json'
      }
    };
    https.get(url, options, (res) => {
      // Follow redirects (3xx)
      if (res.statusCode >= 300 && res.statusCode < 400 && res.headers.location) {
        return resolve(httpsGet(res.headers.location));
      }
      let data = '';
      res.on('data', chunk => data += chunk);
      res.on('end', () => {
        if (res.statusCode !== 200) {
          return reject(new Error(`HTTP ${res.statusCode}: ${data.substring(0, 100)}`));
        }
        try { resolve(JSON.parse(data)); }
        catch (e) { reject(new Error('JSON parse error: ' + e.message)); }
      });
    }).on('error', reject);
  });
}

// ─── Static fallback dataset (used if restcountries.com is unreachable) ───
const COUNTRY_FALLBACK = {
  france: { common: 'France', capital: ['Paris'], currencies: { EUR: { name: 'Euro' } }, languages: { fra: 'French' }, population: 67391582, region: 'Europe', timezones: ['UTC+01:00'], area: 551695 },
  japan: { common: 'Japan', capital: ['Tokyo'], currencies: { JPY: { name: 'Japanese yen' } }, languages: { jpn: 'Japanese' }, population: 125836021, region: 'Asia', timezones: ['UTC+09:00'], area: 377930 },
  india: { common: 'India', capital: ['New Delhi'], currencies: { INR: { name: 'Indian rupee' } }, languages: { hin: 'Hindi' }, population: 1380004385, region: 'Asia', timezones: ['UTC+05:30'], area: 3287263 },
  brazil: { common: 'Brazil', capital: ['Brasília'], currencies: { BRL: { name: 'Brazilian real' } }, languages: { por: 'Portuguese' }, population: 212559417, region: 'Americas', timezones: ['UTC-03:00'], area: 8515767 },
  germany: { common: 'Germany', capital: ['Berlin'], currencies: { EUR: { name: 'Euro' } }, languages: { deu: 'German' }, population: 83783942, region: 'Europe', timezones: ['UTC+01:00'], area: 357114 },
  australia: { common: 'Australia', capital: ['Canberra'], currencies: { AUD: { name: 'Australian dollar' } }, languages: { eng: 'English' }, population: 25499884, region: 'Oceania', timezones: ['UTC+08:00'], area: 7692024 },
  canada: { common: 'Canada', capital: ['Ottawa'], currencies: { CAD: { name: 'Canadian dollar' } }, languages: { eng: 'English' }, population: 37742154, region: 'Americas', timezones: ['UTC-05:00'], area: 9984670 },
  'united kingdom': { common: 'United Kingdom', capital: ['London'], currencies: { GBP: { name: 'British pound' } }, languages: { eng: 'English' }, population: 67886011, region: 'Europe', timezones: ['UTC+00:00'], area: 242900 },
  'united states': { common: 'United States', capital: ['Washington, D.C.'], currencies: { USD: { name: 'United States dollar' } }, languages: { eng: 'English' }, population: 331002651, region: 'Americas', timezones: ['UTC-05:00'], area: 9372610 },
  italy: { common: 'Italy', capital: ['Rome'], currencies: { EUR: { name: 'Euro' } }, languages: { ita: 'Italian' }, population: 60461826, region: 'Europe', timezones: ['UTC+01:00'], area: 301336 }
};

// ─── Build a Lex V2 Close response ───────────────────────────
function buildResponse(message, intentName, slots) {
  return {
    sessionState: {
      dialogAction: { type: 'Close' },
      intent: {
        name: intentName,
        slots: slots || {},
        state: 'Fulfilled'
      }
    },
    messages: [{
      contentType: 'PlainText',
      content: message
    }]
  };
}

// ─── Helper: safely get slot value ───────────────────────────
function getSlot(slots, name) {
  return slots?.[name]?.value?.interpretedValue || null;
}

// ─────────────────────────────────────────────────────────────
// INTENT HANDLERS
// ─────────────────────────────────────────────────────────────

async function handleCountryInfo(slots, intentName) {
  const country = getSlot(slots, 'target_countries');
  const item    = getSlot(slots, 'item');

  if (!country) {
    return buildResponse('Please specify a country name.', intentName, slots);
  }

  let c;
  try {
    const results = await httpsGet(`https://restcountries.com/v3.1/name/${encodeURIComponent(country)}`);
    // restcountries.com returns either an array of country objects, or
    // an object like { status: 404, message: "Not Found" } when nothing matches.
    if (!Array.isArray(results) || results.length === 0 || results.status === 404) {
      throw new Error('No results from restcountries.com');
    }
    c = results[0];
  } catch (err) {
    console.log(`API call failed or returned no results (${err.message}), using fallback data`);
    c = COUNTRY_FALLBACK[country.toLowerCase().trim()];
    if (!c) {
      return buildResponse(
        `Sorry, I couldn't fetch live data for ${country} and don't have it offline. Try: France, Japan, India, Brazil, Germany, Australia, Canada, United Kingdom, United States, or Italy.`,
        intentName, slots
      );
    }
  }

  try {
    const name     = c.name?.common || c.common || country;
    const capital  = c.capital?.[0] || 'N/A';
    const currency = Object.values(c.currencies || {})[0]?.name || 'N/A';
    const language = Object.values(c.languages || {})[0] || 'N/A';
    const pop      = c.population?.toLocaleString() || 'N/A';
    const region   = c.region || 'N/A';
    const timezone = c.timezones?.[0] || 'N/A';
    const area     = c.area ? c.area.toLocaleString() + ' km²' : 'N/A';

    let msg;
    switch ((item || 'all').toLowerCase()) {
      case 'capital':   msg = `The capital of ${name} is ${capital}.`; break;
      case 'currency':  msg = `The currency of ${name} is ${currency}.`; break;
      case 'language':  msg = `The official language of ${name} is ${language}.`; break;
      case 'population':msg = `The population of ${name} is approximately ${pop}.`; break;
      case 'region':    msg = `${name} is located in ${region}.`; break;
      case 'timezone':  msg = `The timezone of ${name} is ${timezone}.`; break;
      case 'area':      msg = `The area of ${name} is ${area}.`; break;
      default:
        msg = `Here is information about ${name}:\n` +
              `• Capital: ${capital}\n• Currency: ${currency}\n• Language: ${language}\n` +
              `• Population: ${pop}\n• Region: ${region}\n• Timezone: ${timezone}\n• Area: ${area}`;
    }
    return buildResponse(msg, intentName, slots);
  } catch (err) {
    return buildResponse(`Error formatting country info: ${err.message}`, intentName, slots);
  }
}

async function handleWeatherInfo(slots, intentName) {
  const city = getSlot(slots, 'city');
  const apiKey = process.env.OPENWEATHER_API_KEY;

  if (!city) return buildResponse('Please specify a city name.', intentName, slots);
  if (!apiKey || apiKey === 'YOUR_KEY') {
    return buildResponse('Weather service is not configured. Please add the OPENWEATHER_API_KEY environment variable.', intentName, slots);
  }

  try {
    const url = `https://api.openweathermap.org/data/2.5/weather?q=${encodeURIComponent(city)}&appid=${apiKey}&units=metric`;
    const data = await httpsGet(url);
    if (data.cod !== 200) {
      return buildResponse(`Could not find weather for ${city}. Please check the city name.`, intentName, slots);
    }
    const desc  = data.weather[0].description;
    const temp  = data.main.temp;
    const feels = data.main.feels_like;
    const humid = data.main.humidity;
    const wind  = data.wind.speed;
    const msg = `Weather in ${city}:\n• Condition: ${desc}\n• Temperature: ${temp}°C (feels like ${feels}°C)\n• Humidity: ${humid}%\n• Wind Speed: ${wind} m/s`;
    return buildResponse(msg, intentName, slots);
  } catch (err) {
    return buildResponse(`Error fetching weather: ${err.message}`, intentName, slots);
  }
}

async function handleCurrencyConversion(slots, intentName) {
  const amount       = parseFloat(getSlot(slots, 'amount') || '1');
  const fromCurrency = (getSlot(slots, 'from_currency') || 'USD').toUpperCase();
  const toCurrency   = (getSlot(slots, 'to_currency')   || 'EUR').toUpperCase();

  try {
    const data = await httpsGet(`https://open.er-api.com/v6/latest/${fromCurrency}`);
    if (data.result !== 'success') {
      return buildResponse(`Could not fetch exchange rates for ${fromCurrency}.`, intentName, slots);
    }
    const rate = data.rates[toCurrency];
    if (!rate) {
      return buildResponse(`Currency ${toCurrency} not found.`, intentName, slots);
    }
    const converted = (amount * rate).toFixed(2);
    const msg = `💱 Currency Conversion:\n${amount} ${fromCurrency} = ${converted} ${toCurrency}\n(Rate: 1 ${fromCurrency} = ${rate.toFixed(4)} ${toCurrency})`;
    return buildResponse(msg, intentName, slots);
  } catch (err) {
    return buildResponse(`Error converting currency: ${err.message}`, intentName, slots);
  }
}

function handleTravelRecommendations(slots, intentName) {
  const budget    = (getSlot(slots, 'budget')    || 'moderate').toLowerCase();
  const interests = (getSlot(slots, 'interests') || 'culture').toLowerCase();

  const recommendations = {
    budget: {
      adventure:   'Vietnam, Nepal, Bolivia — affordable adventure destinations',
      culture:     'Morocco, Cambodia, Egypt — rich culture on a budget',
      relaxation:  'Bali, Thailand, Sri Lanka — budget beach paradises',
      food:        'Mexico, India, Vietnam — amazing food scenes at low cost',
      nature:      'Costa Rica, Peru, South Africa — stunning nature affordably',
      history:     'Greece (off-season), Turkey, Jordan — ancient history affordably',
      shopping:    'Thailand, India, Turkey — great markets at low prices'
    },
    moderate: {
      adventure:   'New Zealand, Iceland, Patagonia — epic adventure experiences',
      culture:     'Japan, Spain, Italy — world-class culture at moderate cost',
      relaxation:  'Maldives (budget resorts), Bali, Portugal — great relaxation',
      food:        'France, Japan, Spain — globally-renowned cuisine',
      nature:      'Canada, Norway, Kenya — spectacular natural beauty',
      history:     'Rome, Athens, Cairo — incredible historical sites',
      shopping:    'Dubai, Singapore, Hong Kong — great shopping destinations'
    },
    luxury: {
      adventure:   'Antarctica, Galapagos, Arctic Norway — ultimate luxury adventures',
      culture:     'Japan (ryokan), France, UAE — luxury cultural experiences',
      relaxation:  'Maldives, Seychelles, Bora Bora — ultimate tropical luxury',
      food:        'France, Tokyo, Copenhagen — Michelin-star dining destinations',
      nature:      'Seychelles, Patagonia, Borneo — luxury eco-experiences',
      history:     'Egypt (private tour), Greece (island hopping), Peru — exclusive history',
      shopping:    'Paris, Milan, New York — world\'s best luxury shopping'
    }
  };

  const rec = recommendations[budget]?.[interests]
    || `Try exploring destinations that match your ${interests} interests with a ${budget} budget!`;

  return buildResponse(
    `✈️ Travel Recommendations for ${budget} budget & ${interests} interests:\n${rec}`,
    intentName, slots
  );
}

function handleFlightStatus(slots, intentName) {
  // Real-time flight APIs require paid subscriptions.
  // This simulation demonstrates the intent flow.
  const flightNumber = getSlot(slots, 'flight_number') || 'Unknown';
  const statuses = ['On Time', 'Delayed by 30 minutes', 'Boarding', 'Departed', 'Landed'];
  const status = statuses[Math.floor(Math.random() * statuses.length)];
  const gates  = ['A12', 'B7', 'C3', 'D15', 'E2'];
  const gate   = gates[Math.floor(Math.random() * gates.length)];
  const msg = `✈️ Flight Status for ${flightNumber.toUpperCase()}:\n• Status: ${status}\n• Gate: ${gate}\n• Last Updated: Just now\n\n(Note: This is a simulated response. Integrate a real flight API for live data.)`;
  return buildResponse(msg, intentName, slots);
}

function handleTravelItinerary(slots, intentName) {
  const destination = getSlot(slots, 'destination') || 'your destination';
  const duration    = parseInt(getSlot(slots, 'duration') || '3', 10);

  const templates = [
    { activity: 'Arrive and check-in. Explore the city center and local markets.', time: 'Morning: Arrival | Afternoon: City Center | Evening: Local Dinner' },
    { activity: 'Visit major landmarks and museums. Take a guided tour.', time: 'Morning: Landmarks | Afternoon: Museum | Evening: Cultural Show' },
    { activity: 'Day trip to nearby attractions. Local food tasting tour.', time: 'Morning: Day Trip | Afternoon: Food Tour | Evening: Leisure' },
    { activity: 'Shopping and souvenir hunting. Relax at local parks.', time: 'Morning: Shopping | Afternoon: Parks | Evening: Farewell Dinner' },
    { activity: 'Explore hidden gems and offbeat spots recommended by locals.', time: 'Morning: Hidden Gems | Afternoon: Local Spots | Evening: Night Market' }
  ];

  let itinerary = `📅 ${duration}-Day Itinerary for ${destination}:\n\n`;
  for (let i = 0; i < Math.min(duration, 7); i++) {
    const t = templates[i % templates.length];
    itinerary += `Day ${i+1}: ${t.time}\n`;
  }
  itinerary += '\nHave a wonderful trip! 🌍';
  return buildResponse(itinerary, intentName, slots);
}

// ─────────────────────────────────────────────────────────────
// MAIN HANDLER
// ─────────────────────────────────────────────────────────────
export const handler = async (event) => {
  const intentName = event?.sessionState?.intent?.name || 'Unknown';
  const slots      = event?.sessionState?.intent?.slots || {};

  console.log(`Intent: ${intentName}`, JSON.stringify(slots));

  switch (intentName) {
    case 'CountryInfo':            return await handleCountryInfo(slots, intentName);
    case 'WeatherInfo':            return await handleWeatherInfo(slots, intentName);
    case 'CurrencyConversion':     return await handleCurrencyConversion(slots, intentName);
    case 'TravelRecommendations':  return handleTravelRecommendations(slots, intentName);
    case 'FlightStatus':           return handleFlightStatus(slots, intentName);
    case 'TravelItinerary':        return handleTravelItinerary(slots, intentName);
    default:
      return buildResponse(`I received intent '${intentName}' but have no handler for it.`, intentName, slots);
  }
};
"""

# Write the Node.js code to a local file
with open("/tmp/index.mjs", "w") as f:
    f.write(LAMBDA_CODE)

print("✅ Lambda function code written to /tmp/index.mjs")

## Cell 5 — Package and Deploy Lambda Function

Lambda requires code to be uploaded as a ZIP file. We create the ZIP in memory and upload it directly via `boto3` — no S3 bucket required.

In [ ]:
# ── Step 1: Zip the Lambda code in memory ──
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("/tmp/index.mjs", "index.mjs")
zip_buffer.seek(0)
zip_bytes = zip_buffer.read()
print(f"✅ ZIP created — size: {len(zip_bytes):,} bytes")

# ── Step 2: Create or update the Lambda function ──
try:
    response = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime="nodejs22.x",
        Role=LAMBDA_ROLE_ARN,
        Handler="index.handler",
        Code={"ZipFile": zip_bytes},
        Description="TravelIntelligenceBot — integrates Lex with third-party APIs",
        Timeout=30,           # 30 seconds — allows time for external API calls
        MemorySize=512,       # 512 MB — handles multiple concurrent API calls
        Architectures=["x86_64"]
    )
    LAMBDA_ARN = response["FunctionArn"]
    print(f"✅ Lambda function CREATED: {LAMBDA_ARN}")

except lambda_client.exceptions.ResourceConflictException:
    # Function already exists — update its code
    response = lambda_client.update_function_code(
        FunctionName=LAMBDA_FUNCTION_NAME,
        ZipFile=zip_bytes
    )
    LAMBDA_ARN = response["FunctionArn"]
    print(f"ℹ️  Lambda function already exists — code UPDATED: {LAMBDA_ARN}")

# Wait for function to be active
print("⏳ Waiting for Lambda to become Active...")
waiter = lambda_client.get_waiter("function_active_v2")
waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)
print("✅ Lambda function is Active and ready.")

## Cell 6 — Configure Environment Variables on Lambda

Equivalent to the **Configuration → Environment Variables** step in the console. The API keys are injected as environment variables so they are never hardcoded in the function source.

In [ ]:
env_vars = {
    "OPENWEATHER_API_KEY": OPENWEATHER_API_KEY,
    "NEWS_API_KEY":        NEWS_API_KEY
}

lambda_client.update_function_configuration(
    FunctionName=LAMBDA_FUNCTION_NAME,
    Environment={"Variables": env_vars},
    Timeout=30,
    MemorySize=512
)

# Wait for configuration update to complete
waiter = lambda_client.get_waiter("function_updated_v2")
waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)

print("✅ Environment variables configured:")
for k in env_vars:
    print(f"   • {k} = {'[SET]' if env_vars[k] != 'YOUR_KEY' else '[NOT SET — placeholder]'}")

## Cell 7 — Test the Lambda Function Directly

This replicates the **Configure Test Event → Test** step from the console. We send a simulated Lex payload to the Lambda function and verify the response.

In [ ]:
# This is the exact test payload from the original lab guide
test_payload = {
    "sessionState": {
        "intent": {
            "name": "CountryInfo",
            "slots": {
                "target_countries": {
                    "value": {
                        "interpretedValue": "France"
                    }
                },
                "item": {
                    "value": {
                        "interpretedValue": "capital"
                    }
                }
            }
        },
        "sessionAttributes": {}
    }
}

invoke_response = lambda_client.invoke(
    FunctionName=LAMBDA_FUNCTION_NAME,
    InvocationType="RequestResponse",
    Payload=json.dumps(test_payload)
)

result = json.loads(invoke_response["Payload"].read())
print("🧪 Lambda Test — CountryInfo (France, capital)")
print("─" * 50)
if "messages" in result:
    print("Bot Response:", result["messages"][0]["content"])
else:
    print(json.dumps(result, indent=2))

## Cell 8 — Create IAM Role for Amazon Lex

Amazon Lex needs a service role to invoke the Lambda function. We also add a **resource-based policy** to Lambda that allows Lex to call it.

In [ ]:
LEX_ROLE_NAME = "TravelIntelligenceBotLexRole"

lex_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lexv2.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Policy that grants Lex permission to invoke our Lambda function
lex_lambda_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["lambda:InvokeFunction"],
        "Resource": LAMBDA_ARN
    }]
}

try:
    role_resp = iam_client.create_role(
        RoleName=LEX_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lex_trust_policy),
        Description="Runtime role for TravelIntelligenceBot Amazon Lex bot"
    )
    LEX_ROLE_ARN = role_resp["Role"]["Arn"]
    print(f"✅ Created Lex role: {LEX_ROLE_ARN}")
except iam_client.exceptions.EntityAlreadyExistsException:
    LEX_ROLE_ARN = iam_client.get_role(RoleName=LEX_ROLE_NAME)["Role"]["Arn"]
    print(f"ℹ️  Lex role already exists. Using: {LEX_ROLE_ARN}")

# Attach inline policy to the Lex role
iam_client.put_role_policy(
    RoleName=LEX_ROLE_NAME,
    PolicyName="LexInvokeLambdaPolicy",
    PolicyDocument=json.dumps(lex_lambda_policy)
)

# Grant Lex permission to invoke Lambda (resource-based policy on Lambda side)
# IMPORTANT: Lex V2 invokes the Lambda from the *bot-alias* ARN
# (arn:aws:lex:<region>:<account>:bot-alias/<BOT_ID>/<ALIAS_ID>), not from
# arn:.../bot/*. We add permissions covering both the specific bot's
# aliases and a wildcard fallback so the test alias (TSTALIASID) is allowed
# even if BOT_ID isn't known yet at this point in the notebook.

permissions_to_add = [
    # Covers any alias of any bot in this account/region (broadest, safest for a lab)
    ("AllowLexInvokeBotAliasWildcard", f"arn:aws:lex:{AWS_REGION}:{ACCOUNT_ID}:bot-alias/*"),
    # Covers the legacy "bot/*" pattern too, in case it's also checked
    ("AllowLexInvokeBotWildcard", f"arn:aws:lex:{AWS_REGION}:{ACCOUNT_ID}:bot/*"),
]

for statement_id, source_arn in permissions_to_add:
    try:
        lambda_client.add_permission(
            FunctionName=LAMBDA_FUNCTION_NAME,
            StatementId=statement_id,
            Action="lambda:InvokeFunction",
            Principal="lexv2.amazonaws.com",
            SourceArn=source_arn
        )
        print(f"✅ Lambda resource policy updated — {statement_id} ({source_arn}).")
    except lambda_client.exceptions.ResourceConflictException:
        print(f"ℹ️  Lambda permission '{statement_id}' already exists.")

print("⏳ Waiting 10 seconds for IAM propagation...")
time.sleep(10)
print("✅ Lex IAM role ready.")

## Cell 9 — Create the Amazon Lex Bot

Equivalent to **Amazon Lex → Create Bot → Traditional → Create a blank bot** in the console.

We create the bot, then create a **locale** (en_US) which is where intents and slot types live.

In [ ]:
# ── Create the Lex Bot ──
try:
    bot_response = lex_client.create_bot(
        botName=BOT_NAME,
        description="Advanced travel assistant with multi-API integration for country info, weather, currency, and travel planning",
        roleArn=LEX_ROLE_ARN,
        dataPrivacy={"childDirected": False},  # COPPA: No
        idleSessionTTLInSeconds=300
    )
    BOT_ID = bot_response["botId"]
    print(f"✅ Lex Bot created. Bot ID: {BOT_ID}")
except Exception as e:
    if "already exists" in str(e).lower() or "ConflictException" in type(e).__name__:
        # Bot exists — find it by name
        bots = lex_client.list_bots(filters=[{"name": "BotName", "values": [BOT_NAME], "operator": "EQ"}])
        BOT_ID = bots["botSummaries"][0]["botId"]
        print(f"ℹ️  Bot already exists. Bot ID: {BOT_ID}")
    else:
        raise

# Wait for bot to become Available
print("⏳ Waiting for bot to become Available...")
for _ in range(20):
    status = lex_client.describe_bot(botId=BOT_ID)["botStatus"]
    if status == "Available":
        print("✅ Bot status: Available")
        break
    time.sleep(5)
else:
    print(f"⚠️  Bot status after waiting: {status}")

## Cell 10 — Create Bot Version and Locale (en_US)

Before we can add intents, we need a **bot locale** (language configuration). The locale specifies the NLU confidence threshold and which language the bot understands.

In [ ]:
BOT_VERSION = "DRAFT"   # We work on the DRAFT version during development

try:
    lex_client.create_bot_locale(
        botId=BOT_ID,
        botVersion=BOT_VERSION,
        localeId=LOCALE_ID,
        description="English US locale for TravelIntelligenceBot",
        nluIntentConfidenceThreshold=0.40,  # Confidence threshold for intent matching
        voiceSettings={"voiceId": "Joanna"}  # Optional TTS voice
    )
    print(f"✅ Bot locale '{LOCALE_ID}' created.")
except (lex_client.exceptions.ConflictException, lex_client.exceptions.PreconditionFailedException) as e:
    # Lex V2 returns PreconditionFailedException (instead of ConflictException)
    # with "already exists" when the locale was created in a previous run.
    if isinstance(e, lex_client.exceptions.PreconditionFailedException) and "already exists" not in str(e):
        raise
    print(f"ℹ️  Locale '{LOCALE_ID}' already exists.")

# Confirm locale status
locale_info = lex_client.describe_bot_locale(botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID)
print(f"   Locale status: {locale_info['botLocaleStatus']}")

## Cell 11 — Create Custom Slot Types

Equivalent to **Bot Configuration → Slot types → Add blank slot type** in the console.

We create three custom slot types:
- **InfoType** — country info categories (capital, currency, language, etc.)
- **BudgetRange** — travel budget tiers
- **TravelInterests** — traveller interest categories

In [ ]:
def create_slot_type(name, description, values):
    """
    Creates a custom slot type in Lex V2.
    - name: slot type name
    - description: short description
    - values: list of string values for this slot type
    Returns the slotTypeId.
    """
    slot_values = [
        {
            "sampleValue": {"value": v}
            # NOTE: "synonyms" omitted entirely. Lex V2 requires synonyms,
            # if provided, to be a non-empty list (min length 1), so we
            # don't include the key at all to avoid a ParamValidationError.
        } for v in values
    ]
    try:
        resp = lex_client.create_slot_type(
            slotTypeName=name,
            description=description,
            slotTypeValues=slot_values,
            valueSelectionSetting={"resolutionStrategy": "OriginalValue"},
            botId=BOT_ID,
            botVersion=BOT_VERSION,
            localeId=LOCALE_ID
        )
        print(f"   ✅ Created slot type '{name}' (ID: {resp['slotTypeId']})")
        return resp["slotTypeId"]
    except (
        lex_client.exceptions.ConflictException,
        lex_client.exceptions.ValidationException,
        lex_client.exceptions.PreconditionFailedException,
    ) as e:
        # Lex V2 may return ValidationException or PreconditionFailedException
        # (instead of ConflictException) with "already exists" in the message
        # when the slot type name is already taken — handle all three the same way.
        if not isinstance(e, lex_client.exceptions.ConflictException) and "already exists" not in str(e):
            raise
        # Already exists — look up the ID
        existing = lex_client.list_slot_types(
            botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID,
            filters=[{"name": "SlotTypeName", "values": [name], "operator": "EQ"}]
        )
        slot_id = existing["slotTypeSummaries"][0]["slotTypeId"]
        print(f"   ℹ️  Slot type '{name}' already exists (ID: {slot_id})")
        return slot_id

print("Creating custom slot types...")

INFO_TYPE_ID = create_slot_type(
    "InfoType",
    "Type of country information requested",
    ["capital", "currency", "language", "population", "region", "timezone", "area", "all"]
)

BUDGET_RANGE_ID = create_slot_type(
    "BudgetRange",
    "Travel budget categories",
    ["budget", "moderate", "luxury", "backpacker"]
)

TRAVEL_INTERESTS_ID = create_slot_type(
    "TravelInterests",
    "Travel interest categories",
    ["adventure", "culture", "relaxation", "food", "nature", "history", "shopping"]
)

print("\n✅ All custom slot types created.")

## Cell 12 — Helper Functions for Intent Creation

We define reusable helpers to add intents and their slots programmatically.

In [ ]:
def create_intent(name, description, sample_utterances):
    """
    Creates an intent in Lex V2 with sample utterances and Lambda fulfillment.
    Returns the intentId.
    """
    utterances = [{"utterance": u} for u in sample_utterances]

    # Fulfillment code hook — calls our Lambda function when intent is triggered
    fulfillment_hook = {
        "enabled": True,
        "postFulfillmentStatusSpecification": {
            "successResponse": {
                "messageGroups": [{"message": {"plainTextMessage": {"value": "Request processed."}}}],
                "allowInterrupt": True
            },
            "failureResponse": {
                "messageGroups": [{"message": {"plainTextMessage": {"value": "Sorry, something went wrong."}}}],
                "allowInterrupt": True
            },
            "timeoutResponse": {
                "messageGroups": [{"message": {"plainTextMessage": {"value": "Request timed out."}}}],
                "allowInterrupt": True
            }
        }
    }

    try:
        resp = lex_client.create_intent(
            intentName=name,
            description=description,
            sampleUtterances=utterances,
            fulfillmentCodeHook=fulfillment_hook,
            botId=BOT_ID,
            botVersion=BOT_VERSION,
            localeId=LOCALE_ID
        )
        print(f"   ✅ Intent '{name}' created (ID: {resp['intentId']})")
        return resp["intentId"]
    except (
        lex_client.exceptions.ConflictException,
        lex_client.exceptions.ValidationException,
        lex_client.exceptions.PreconditionFailedException,
    ) as e:
        # Lex V2 sometimes returns ValidationException or
        # PreconditionFailedException (instead of ConflictException) with
        # "already exists" in the message when the intent name is taken —
        # handle all three the same way.
        if not isinstance(e, lex_client.exceptions.ConflictException) and "already exists" not in str(e):
            raise  # re-raise if it's a different validation error
        existing = lex_client.list_intents(
            botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID,
            filters=[{"name": "IntentName", "values": [name], "operator": "EQ"}]
        )
        intent_id = existing["intentSummaries"][0]["intentId"]
        print(f"   ℹ️  Intent '{name}' already exists (ID: {intent_id})")
        return intent_id


def add_slot(intent_id, slot_name, slot_type_id, prompt, required=True, is_builtin=False):
    """
    Adds a slot to an existing intent.
    - slot_type_id: for built-in types, pass the full ARN string like 'AMAZON.Country'
    - is_builtin: set True for AMAZON.* built-in slot types
    """
    value_elicitation = {
        "promptSpecification": {
            "messageGroups": [{"message": {"plainTextMessage": {"value": prompt}}}],
            "maxRetries": 3,
            "allowInterrupt": True
        },
        "slotConstraint": "Required" if required else "Optional"
    }

    # Build slot type spec based on whether it's a built-in or custom type
    if is_builtin:
        slot_type_spec = {"slotTypeId": slot_type_id}
    else:
        slot_type_spec = {"slotTypeId": slot_type_id}

    try:
        resp = lex_client.create_slot(
            slotName=slot_name,
            slotTypeId=slot_type_id,
            valueElicitationSetting=value_elicitation,
            intentId=intent_id,
            botId=BOT_ID,
            botVersion=BOT_VERSION,
            localeId=LOCALE_ID
        )
        req_label = "Required" if required else "Optional"
        print(f"      ↳ Slot '{slot_name}' ({req_label}) added.")
        return resp["slotId"]
    except (
        lex_client.exceptions.ConflictException,
        lex_client.exceptions.ValidationException,
        lex_client.exceptions.PreconditionFailedException,
    ) as e:
        # Lex V2 may raise ValidationException OR PreconditionFailedException
        # (instead of ConflictException) with "already exists" for duplicate slots.
        if not isinstance(e, lex_client.exceptions.ConflictException) and "already exists" not in str(e):
            raise
        print(f"      ↳ Slot '{slot_name}' already exists.")

print("✅ Helper functions defined.")

## Cell 13 — Intent 1: CountryInfo

Handles queries like *"What is the capital of France?"* or *"Tell me about India."*

In [ ]:
print("Creating Intent 1: CountryInfo")

COUNTRY_INFO_ID = create_intent(
    name="CountryInfo",
    description="Provides information about a country such as capital, currency, language, and more",
    sample_utterances=[
        "Tell me about {target_countries}",
        "{target_countries} information",
        "Information about {target_countries}",
        "What is the {item} of {target_countries}",
        "{item} of {target_countries}",
        "{target_countries} {item}",
        "Show me {item} for {target_countries}",
        "I want to know about {target_countries}"
    ]
)

# Slot 1: target_countries — Required. Uses built-in AMAZON.Country type
add_slot(
    intent_id=COUNTRY_INFO_ID,
    slot_name="target_countries",
    slot_type_id="AMAZON.Country",
    prompt="Which country would you like to know about?",
    required=True,
    is_builtin=True
)

# Slot 2: item — Optional. Uses custom InfoType slot type
# Not required: user can say "Tell me about France" and get all info
add_slot(
    intent_id=COUNTRY_INFO_ID,
    slot_name="item",
    slot_type_id=INFO_TYPE_ID,
    prompt="What would you like to know? You can ask about capital, currency, language, population, region, timezone, area, or all information.",
    required=False
)

print("✅ CountryInfo intent complete.")

## Cell 14 — Intent 2: WeatherInfo

Handles queries like *"What is the weather in Tokyo?"* — calls OpenWeatherMap API via Lambda.

In [ ]:
print("Creating Intent 2: WeatherInfo")

WEATHER_INFO_ID = create_intent(
    name="WeatherInfo",
    description="Provides current weather information for a specified city",
    sample_utterances=[
        "What is the weather in {city}",
        "Weather for {city}",
        "How is the weather in {city}",
        "Is it raining in {city}",
        "Temperature in {city}",
        "{city} weather",
        "Weather in {city}",
        "Tell me the weather for {city}",
        "What's the weather like in {city}",
        "Check the weather for {city}"
    ]
)

add_slot(
    intent_id=WEATHER_INFO_ID,
    slot_name="city",
    slot_type_id="AMAZON.City",
    prompt="Which city's weather would you like to know?",
    required=True,
    is_builtin=True
)

print("✅ WeatherInfo intent complete.")

## Cell 15 — Intent 3: CurrencyConversion

Handles queries like *"Convert 100 USD to EUR"* — calls the ExchangeRate API via Lambda.

In [ ]:
print("Creating Intent 3: CurrencyConversion")

CURRENCY_ID = create_intent(
    name="CurrencyConversion",
    description="Converts an amount from one currency to another using live exchange rates",
    sample_utterances=[
        "Convert {amount} {from_currency} to {to_currency}",
        "How much is {amount} {from_currency} in {to_currency}",
        "{amount} {from_currency} to {to_currency}",
        "Exchange {from_currency} to {to_currency}",
        "{from_currency} to {to_currency}",
        "Convert {from_currency} to {to_currency}",
        "Change {amount} {from_currency} to {to_currency}",
        "What is {amount} {from_currency} in {to_currency}"
    ]
)

add_slot(CURRENCY_ID, "amount",        "AMAZON.Number",        "How much would you like to convert?", required=True, is_builtin=True)
add_slot(CURRENCY_ID, "from_currency", "AMAZON.AlphaNumeric",  "Which currency are you converting from? (e.g., USD, EUR, GBP)", required=True, is_builtin=True)
add_slot(CURRENCY_ID, "to_currency",   "AMAZON.AlphaNumeric",  "Which currency do you want to convert to? (e.g., JPY, INR, CAD)", required=True, is_builtin=True)

print("✅ CurrencyConversion intent complete.")

## Cell 16 — Intent 4: TravelRecommendations

Handles queries like *"Recommend budget destinations for culture."*

In [ ]:
print("Creating Intent 4: TravelRecommendations")

TRAVEL_REC_ID = create_intent(
    name="TravelRecommendations",
    description="Recommends travel destinations based on budget and interests",
    sample_utterances=[
        "Recommend travel destinations",
        "Suggest places for {interests}",
        "Where should I travel with {budget} budget",
        "Travel recommendations for {budget}",
        "Best {interests} destinations",
        "Give me travel suggestions",
        "I want {interests} travel recommendations",
        "Suggest {budget} destinations",
        "Where should I go for {interests}",
        "Travel ideas for {budget} budget"
    ]
)

add_slot(TRAVEL_REC_ID, "budget",    BUDGET_RANGE_ID,      "What's your budget? Choose from: budget, moderate, or luxury.", required=True)
add_slot(TRAVEL_REC_ID, "interests", TRAVEL_INTERESTS_ID,  "What are you interested in? Options: adventure, culture, relaxation, food, nature, history, or shopping.", required=True)

print("✅ TravelRecommendations intent complete.")

## Cell 17 — Intent 5: FlightStatus

Handles queries like *"Flight status for BA456."*

> **Note:** Real-time flight data APIs (FlightAware, AviationStack) require paid subscriptions. The Lambda function returns a simulated response to demonstrate the intent flow.

In [ ]:
print("Creating Intent 5: FlightStatus")

FLIGHT_ID = create_intent(
    name="FlightStatus",
    description="Checks the status of a flight by flight number",
    sample_utterances=[
        "Flight status for {flight_number}",
        "Check flight {flight_number}",
        "Is flight {flight_number} on time",
        "Status of {flight_number}",
        "{flight_number} status",
        "Track flight {flight_number}",
        "Flight {flight_number}",
        "What is the status of flight {flight_number}",
        "Check status for {flight_number}"
    ]
)

add_slot(FLIGHT_ID, "flight_number", "AMAZON.AlphaNumeric", "What is your flight number? (e.g., AA123, BA456)", required=True, is_builtin=True)

print("✅ FlightStatus intent complete.")

## Cell 18 — Intent 6: TravelItinerary

Handles queries like *"Create 3 day itinerary for Paris."*

In [ ]:
print("Creating Intent 6: TravelItinerary")

ITINERARY_ID = create_intent(
    name="TravelItinerary",
    description="Creates a day-by-day travel itinerary for a destination",
    sample_utterances=[
        "Create itinerary for {destination}",
        "Plan {duration} day trip to {destination}",
        "{duration} days in {destination}",
        "Itinerary for {destination}",
        "Make a {duration} day plan for {destination}",
        "Plan my trip to {destination}",
        "{destination} itinerary",
        "Create {duration} day itinerary for {destination}",
        "Plan a trip to {destination} for {duration} days"
    ]
)

add_slot(ITINERARY_ID, "destination", "AMAZON.City",   "Which destination would you like an itinerary for?", required=True, is_builtin=True)
add_slot(ITINERARY_ID, "duration",    "AMAZON.Number", "How many days will you be traveling? (e.g., 3, 5, 7)", required=True, is_builtin=True)

print("✅ TravelItinerary intent complete.")
print("\n🎉 All 6 intents created successfully!")

## Cell 19 — Configure Lambda Integration on the Bot Alias

In the console, this is done under **Aliases → TestBotAlias → Languages → English → Lambda function**.

We first build the bot (to create the `TestBotAlias`), then attach the Lambda function to it.

In [ ]:
# ── Fix: Set slot priorities for every intent ──
# Lex V2 requires every slot in an intent to have a "priority" defined
# (the order in which Lex elicits missing slots from the user). create_slot()
# does NOT set this automatically, so the build fails with errors like:
#   "Slot ids [...] in intent ... don't define a slot priority."
# Here we fetch every intent in this locale, list its slots, and call
# update_intent with a slotPriorities list (in the order the slots were
# created) before attempting to build the bot.
print("🔧 Setting slot priorities for all intents...")

intents_resp = lex_client.list_intents(
    botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID, maxResults=50
)

for intent_summary in intents_resp["intentSummaries"]:
    intent_id = intent_summary["intentId"]
    intent_name = intent_summary["intentName"]

    # Skip the built-in FallbackIntent — it has no slots
    if intent_name == "FallbackIntent":
        continue

    # List slots for this intent
    slots_resp = lex_client.list_slots(
        botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID, intentId=intent_id, maxResults=50
    )
    slot_summaries = slots_resp.get("slotSummaries", [])
    if not slot_summaries:
        continue

    # Assign sequential priorities (1, 2, 3, ...) in the order returned
    slot_priorities = [
        {"priority": i + 1, "slotId": s["slotId"]}
        for i, s in enumerate(slot_summaries)
    ]

    # Fetch the full intent definition (required by update_intent, since it
    # is a full PUT — fields not passed get reset to defaults)
    full_intent = lex_client.describe_intent(
        botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID, intentId=intent_id
    )

    update_kwargs = {
        "intentId": intent_id,
        "intentName": full_intent["intentName"],
        "botId": BOT_ID,
        "botVersion": BOT_VERSION,
        "localeId": LOCALE_ID,
        "slotPriorities": slot_priorities,
    }
    # Preserve existing fields if present
    for key in ["description", "sampleUtterances", "fulfillmentCodeHook",
                 "dialogCodeHook", "intentClosingSetting", "intentConfirmationSetting"]:
        if key in full_intent:
            update_kwargs[key] = full_intent[key]

    lex_client.update_intent(**update_kwargs)
    slot_names = ", ".join(s["slotName"] for s in slot_summaries)
    print(f"   ✅ Set slot priorities for intent '{intent_name}': {slot_names}")

print("✅ Slot priorities configured for all intents.\n")

# ── Build the Bot ──
print("🔨 Building the bot (this may take 2-5 minutes)...")

build_resp = lex_client.build_bot_locale(
    botId=BOT_ID,
    botVersion=BOT_VERSION,
    localeId=LOCALE_ID
)
print(f"   Build initiated. Status: {build_resp['botLocaleStatus']}")

# Poll until build completes
for attempt in range(40):
    locale_status = lex_client.describe_bot_locale(
        botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID
    )["botLocaleStatus"]
    print(f"   [{attempt+1}] Locale status: {locale_status}")
    if locale_status == "Built":
        print("✅ Bot locale built successfully!")
        break
    elif locale_status in ["Failed", "Deleting"]:
        print(f"❌ Build failed with status: {locale_status}")
        break
    time.sleep(15)
else:
    print("⚠️  Build is taking longer than expected. Check the Lex console.")

In [ ]:
# ── Find the TestBotAlias and attach Lambda to it ──
aliases = lex_client.list_bot_aliases(botId=BOT_ID)
test_alias = next(
    (a for a in aliases["botAliasSummaries"] if a["botAliasName"] == "TestBotAlias"),
    None
)

if not test_alias:
    print("❌ TestBotAlias not found. The bot may not have built successfully yet.")
else:
    ALIAS_ID = test_alias["botAliasId"]
    print(f"✅ Found TestBotAlias. Alias ID: {ALIAS_ID}")

    # Attach Lambda to the alias for en_US locale
    lex_client.update_bot_alias(
        botId=BOT_ID,
        botAliasId=ALIAS_ID,
        botAliasName="TestBotAlias",
        botVersion=BOT_VERSION,
        botAliasLocaleSettings={
            LOCALE_ID: {
                "enabled": True,
                "codeHookSpecification": {
                    "lambdaCodeHook": {
                        "lambdaARN": LAMBDA_ARN,
                        "codeHookInterfaceVersion": "1.0"
                    }
                }
            }
        }
    )
    print(f"✅ Lambda function '{LAMBDA_FUNCTION_NAME}' attached to TestBotAlias.")

    # Short wait for alias update
    time.sleep(5)

## Cell 20 — Test the Bot (All 6 Intents)

This replicates the **Test the bot** section in the console. We use the **Lex V2 Runtime** client to send text messages and receive responses — just like a real user would.

Each test corresponds to one of the lab's test conversations.

In [ ]:
import uuid

def chat_with_bot(message, session_id=None):
    """
    Sends a text message to the Lex bot and returns the response.
    Uses the Lex V2 Runtime client (lexv2-runtime).
    """
    if session_id is None:
        session_id = str(uuid.uuid4())

    response = lex_runtime.recognize_text(
        botId=BOT_ID,
        botAliasId=ALIAS_ID,
        localeId=LOCALE_ID,
        sessionId=session_id,
        text=message
    )

    # Extract the bot's reply messages
    messages = response.get("messages", [])
    intent   = response.get("sessionState", {}).get("intent", {}).get("name", "Unknown")
    return {
        "intent": intent,
        "reply": "\n".join(m["content"] for m in messages if m.get("contentType") == "PlainText")
    }

print("✅ chat_with_bot() function ready. Run the next cell to test all 6 conversations.")

In [ ]:
# ── 6 Test Conversations from the original lab ──
test_conversations = [
    ("Test 1 — CountryInfo",          "What is the capital of Australia"),
    ("Test 2 — WeatherInfo",          "Weather in Tokyo"),
    ("Test 3 — CurrencyConversion",   "Convert 100 USD to EUR"),
    ("Test 4 — TravelRecommendations","Recommend budget destinations for culture"),
    ("Test 5 — FlightStatus",         "Flight status for BA456"),
    ("Test 6 — TravelItinerary",      "Create 3 day itinerary for Paris"),
]

print("=" * 60)
print(" CHATBOT TEST RESULTS")
print("=" * 60)

for label, message in test_conversations:
    print(f"\n📌 {label}")
    print(f"   You: {message}")
    try:
        result = chat_with_bot(message)
        print(f"   Intent matched: {result['intent']}")
        print(f"   Bot: {result['reply']}")
    except Exception as e:
        print(f"   ❌ Error: {e}")
    print("-" * 60)

## Cell 21 — Interactive Chat (Optional)

Run this cell to have a live conversation with your bot directly from the notebook. Type `quit` to exit.

In [ ]:
session_id = str(uuid.uuid4())  # Maintain session across turns

print("🤖 TravelIntelligenceBot is ready! Type your message below.")
print("   Try: 'What is the currency of Japan?' or 'Weather in London'")
print("   Type 'quit' to exit.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        break

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye! Safe travels! ✈️")
        break

    try:
        result = chat_with_bot(user_input, session_id=session_id)
        print(f"Bot [{result['intent']}]: {result['reply']}\n")
    except Exception as e:
        print(f"Error: {e}\n")

## Cell 22 — Summary: What We Built

| Component | Details |
|---|---|
| Lambda Function | `TravelIntelligenceBot` — Node.js 22.x, 512 MB, 30s timeout |
| APIs Integrated | restcountries.com, OpenWeatherMap, ExchangeRate-API |
| Lex Bot | `TravelIntelligenceBot` — en_US locale |
| Custom Slot Types | InfoType, BudgetRange, TravelInterests |
| Intents | CountryInfo, WeatherInfo, CurrencyConversion, TravelRecommendations, FlightStatus, TravelItinerary |

### Skills Gained
- Programmatic Lex V2 bot creation using **boto3**
- Lambda deployment and environment variable management via Python
- Intent, slot, and slot-type configuration as code
- End-to-end bot testing via the **Lex V2 Runtime** API

---
Proceed to the next cell to **clean up all resources** created in this lab.

## Cell 23 — Cleanup (Delete All Resources)

> ⚠️ **Run this cell when you are done with the lab** to avoid any ongoing charges.

This deletes, in order:
1. The Lex bot (and all its versions, aliases, locales)
2. The Lambda function
3. The IAM roles created for this lab

In [ ]:
# ── Safety check ──
confirm = input("Type 'DELETE' to confirm deletion of all lab resources: ").strip()
if confirm != "DELETE":
    print("Cleanup cancelled. No resources were deleted.")
else:
    print("\n🗑️  Starting cleanup...")

    # 1. Delete Lex Bot
    try:
        lex_client.delete_bot(botId=BOT_ID, skipResourceInUseCheck=True)
        print("   ✅ Lex bot deletion initiated.")
        time.sleep(10)  # Give Lex time to cascade-delete resources
    except Exception as e:
        print(f"   ⚠️  Lex bot deletion error: {e}")

    # 2. Delete Lambda Function
    try:
        lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
        print("   ✅ Lambda function deleted.")
    except Exception as e:
        print(f"   ⚠️  Lambda deletion error: {e}")

    # 3. Remove inline policy from Lex role, then delete Lex role
    try:
        iam_client.delete_role_policy(RoleName=LEX_ROLE_NAME, PolicyName="LexInvokeLambdaPolicy")
        iam_client.delete_role(RoleName=LEX_ROLE_NAME)
        print("   ✅ Lex IAM role deleted.")
    except Exception as e:
        print(f"   ⚠️  Lex role deletion error: {e}")

    # 4. Detach policy and delete Lambda role
    try:
        iam_client.detach_role_policy(
            RoleName=LAMBDA_ROLE_NAME,
            PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
        )
        iam_client.delete_role(RoleName=LAMBDA_ROLE_NAME)
        print("   ✅ Lambda IAM role deleted.")
    except Exception as e:
        print(f"   ⚠️  Lambda role deletion error: {e}")

    print("\n✅ Cleanup complete! All lab resources have been deleted.")

## Troubleshooting

Refer to the table below if you encounter errors during the lab.

| Error | Cause | Fix |
|---|---|---|
| `InvalidSlotReferenceError` | Slot name in utterance doesn't match defined slot | Ensure slot names in utterances exactly match slot names added to the intent |
| `LanguageBuildFailureError` | Intent fulfillment misconfigured | Check that all intents have `fulfillmentCodeHook` enabled with proper response specs |
| `InvalidSlotToElicitError` | `ElicitSlot` references a non-required or missing slot | Remove the ElicitSlot action, or ensure the target slot is marked Required |
| `MissingBotAliasIdError` | Bot not built yet | Run Cell 19 to build the bot before testing |
| `AccessDeniedException` on Lambda | IAM role not propagated | Wait 15-30 seconds and retry; IAM changes are eventually consistent |
| `ResourceConflictException` | Resource already exists | Safe to ignore — the notebook handles this and reuses the existing resource |
| Weather returns "not configured" | `OPENWEATHER_API_KEY` not set | Update Cell 2 with your real API key and re-run Cells 2 and 6 |

> **Common issue in SageMaker:** If you see `AccessDenied` when calling Lex APIs, ensure your SageMaker execution role has the `AmazonLexFullAccess` managed policy attached. You can add it in the IAM Console under your notebook's execution role.